# Data Ingestion & Parsing

### 🎯 What I’m Learning

In this part of the project, I’m learning how to take information from different sources and prepare it properly for a RAG system. The goal is not just to load files, but to understand what happens to the data before it reaches the retrieval pipeline.

I’m working with:

- Text files and documents
- PDFs and Word files
- CSV and Excel data
- JSON and other structured formats
- Web content
- SQL databases
- Audio/video transcripts
- Different parsing and ingestion approaches
- Common problems that appear with real-world data

The main focus is on **understanding the ingestion process, handling messy data, preserving useful metadata, and preparing clean documents for the next stages of a RAG pipeline.**


# Intro to Data Ingestion

Before a RAG system can retrieve useful information, the raw data first needs to be brought into the system in a consistent and usable form.

In this section, I’m starting with the fundamentals of data ingestion and document representation. The goal is to understand what happens between a raw file and the document object that will eventually be processed for chunking, embeddings, and retrieval.

I’ll begin with simple text files and then gradually explore how LangChain represents documents, how metadata is preserved, how individual and multiple files can be loaded, and how the loaded content can later be split into smaller pieces.

### What I want to understand

- What does "data ingestion" actually mean in a RAG pipeline?
- How does LangChain represent an ingested document?
- What is the difference between `page_content` and `metadata`?
- Why is metadata important for retrieval and filtering?
- How do we load one file versus an entire directory?
- What happens to the source information during ingestion?
- Why do we need to split documents before working with embeddings and retrieval?

The main idea I’m following throughout this notebook is:

**Raw Data → Document → Clean/Structured Content → Chunks → Embeddings → Retrieval**

For now, I’m focusing only on the **ingestion and early preprocessing stages**.


# Intro to Data Ingestion

In this part, I’m learning how to take data from different sources and prepare it for a RAG system.

I’m starting with simple files first and then moving towards more complex data like PDFs, Word files, Excel, JSON, websites, databases, and transcripts.

The main thing I want to understand is what happens to the data before it goes into chunking, embeddings, and retrieval.

I’m also looking at how different loaders work, how metadata is stored, and how the way we prepare the data can affect the RAG pipeline.


In [1]:
import os
from typing import List, Dict, Any
import pandas as pd

## Setting Up the Basic Environment

Before starting the actual examples, I’m importing the main LangChain classes that I’ll use in this notebook.

For now, I’m mainly interested in the Document class and a few text splitters because these will be used again in the next steps.


In [3]:
from langchain_core.documents import Document

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("Set up Completed!")

Set up Completed!


## Introduction to Data Ingestion

Before a RAG system can search anything, we first need to get the data into the system.

Data can come from many different places like text files, PDFs, Word files, databases, websites, or JSON files.

So the first step is usually to load the data and convert it into a format that we can work with.

I’m starting with the basic Document structure first because the same idea will be used with different types of files later.


In [5]:
doc = Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Abhishek Upadhayay",
        "date_created": "2026-09-25",
        "custom_field": "any_value"
    }
)

print("Document Structure")

print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

print("\n📝Metadata is useful for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Content: This is the main text content that will be embedded and searched.
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'Abhishek Upadhayay', 'date_created': '2026-09-25', 'custom_field': 'any_value'}

📝Metadata is useful for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


## Checking the Document Type

I also want to see what type of object LangChain has created here.

This helps me understand that the document is not just a normal string. It is a structured object that contains the actual text along with metadata.


In [7]:
print(type(doc))

<class 'langchain_core.documents.base.Document'>


## Text Files (.txt)

I’m starting with text files because they are the simplest type of data to work with.

There is not much structure here, so it is easier to understand the basic flow first.

I’m creating two small text files so I can load them later and see how LangChain handles them.


In [8]:
import os

os.makedirs("data/course_samples/text_files", exist_ok=True)

## Creating Some Sample Text Files

I’m creating a couple of small text files for testing.

This makes it easier to experiment with different loaders without depending on external files.

Later, I can use the same idea with real documents.


In [10]:
sample_texts = {
    "data/course_samples/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "data/course_samples/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems"""
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


## TextLoader - Reading a Single File

Now I’m loading one text file using TextLoader.

I want to check three things here:

- How many documents were created
- What text was loaded
- What metadata was added

This helps me understand what a simple file looks like after ingestion.


In [30]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "data/course_samples/text_files/python_intro.txt",
    encoding="utf-8"
)

documents = loader.load()

print(f"📄Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

📄Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Metadata: {'source': 'data/course_samples/text_files/python_intro.txt'}


### What I noticed

The text file is converted into a LangChain Document.

The actual text is stored in `page_content`, and the file information is stored in `metadata`.

This same structure will be useful later when the documents are split into chunks and stored for retrieval.


## DirectoryLoader - Reading Multiple Files

Loading one file is easy, but a real RAG system will usually have many files.

So here I’m using DirectoryLoader to load multiple text files from the same folder.

I’m also using a glob pattern so I can control which files should be loaded.

This is useful when I have a whole folder of documents instead of just one file.


In [16]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "data/course_samples/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()

print(f"📁Loaded {len(documents)} documents")

for i, doc in enumerate(documents):
    print(f"\nDocument {i + 1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")


# 📊 Analysis
print("\n📊DirectoryLoader Characteristics:")

print("✅Advantages:")
print("- Loads multiple files at once")
print("- Supports glob patterns")
print("- Shows progress")
print("- Can scan folders recursively")

print("\nThings to keep in mind:")
print("\n❌ Disadvantages:")
print("- All files must be same type")
print("- Limited error handling per file")
print("- Can be memory intensive for large directories")
print("- Files usually need to work with the same loader")
print("- Error handling can become important with many files")
print("- Large folders may use more memory")

100%|██████████| 2/2 [00:00<00:00, 852.59it/s]

📁Loaded 2 documents

Document 1:
  Source: data\course_samples\text_files\machine_learning.txt
  Length: 568 characters

Document 2:
  Source: data\course_samples\text_files\python_intro.txt
  Length: 489 characters

📊DirectoryLoader Characteristics:
✅Advantages:
- Loads multiple files at once
- Supports glob patterns
- Shows progress
- Can scan folders recursively

Things to keep in mind:

❌ Disadvantages:
- All files must be same type
- Limited error handling per file
- Can be memory intensive for large directories
- Files usually need to work with the same loader
- Error handling can become important with many files
- Large folders may use more memory


### What I noticed

DirectoryLoader makes it much easier to work with multiple files.

It also keeps the source information for each document.

This becomes important later because when a chunk is retrieved, I still want to know where that chunk came from.


## Text Splitting Strategies

Now I have the documents loaded, but I still cannot simply send a very large document into the next stage.

I need to divide the text into smaller pieces called chunks.

The way I create these chunks can change the final result, so I want to compare a few different approaches.

For this experiment, I’m looking at:

- Character-based splitting
- Recursive character splitting
- Token-based splitting

I’m using the same text for each method so the comparison is fair.


In [17]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\course_samples\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'), Document(metadata={'source': 'data\\course_samples\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of 

## Getting the Text

Before trying the different splitters, I’m taking the text from one of the loaded documents.

This gives me the same input text for all the experiments below.


In [18]:
# Method 1- Character Text Splitter
text = documents[0].page_content

text

'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'

## 1. Character-Based Splitting

The first method I’m trying is CharacterTextSplitter.

Here the chunk size is based on characters.

I can also control the overlap between chunks.

I’m starting with a space as the separator so I can see how the text gets divided.


In [19]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator=" ",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1️⃣ CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


### Looking at the Output

I’m printing the first two chunks so I can actually see where the splitter is breaking the text.

This is more useful than only checking the number of chunks because I want to understand whether the split is happening at a useful place.


In [20]:
print(char_chunks[0])
print("------------------")
print(char_chunks[1])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
------------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:


## Trying a Different Separator

## 1. Character-Based Splitting

Now I’m changing the separator from a space to a new line.

I’m doing this to see how a small change in the splitting rule can change the final chunks.

This is important because chunk boundaries can affect the quality of retrieval later.


In [22]:
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

char_chunks = char_splitter.split_text(text)

print("CHARACTER TEXT SPLITTER")
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

CHARACTER TEXT SPLITTER
Created 4 chunks
First chunk: Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems...


### Comparing the New Chunks

Now I’m printing a few chunks again so I can compare them with the previous result.

The goal here is simply to see how the separator changes the way the text is broken.


In [23]:
print(char_chunks[0])
print("-------------")
print(char_chunks[1])
print("-------------")
print(char_chunks[2])

Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
-------------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
Types of Machine Learning:
-------------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties


## 2. Recursive Character Splitting

Now I’m trying RecursiveCharacterTextSplitter.

Instead of depending on only one separator, I can give it multiple separators and let it try them one by one.

This can be useful when the document has different types of structure.

I want to see how this behaves compared with the simple character splitter.


In [21]:
# Method 2: Recursive character splitting (RECOMMENDED)
print("\n2️⃣ RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")


2️⃣ RECURSIVE CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


### Looking at the Recursive Chunks

I’m printing a few chunks here so I can see where the recursive splitter is making the breaks.

For now, I’m mainly interested in how the chunks look and how they compare with the previous method.


In [24]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("------------------")
print(recursive_chunks[2])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
-----------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:
------------------
Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


## Testing a Simple Example

I also want to test the splitter with a different piece of text.

This time I’m using a simple paragraph and giving the splitter only a limited separator.

I’m doing this to understand what happens when the text does not have many natural places where it can be split.


In [26]:
simple_text = (
    "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."
)

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"Simple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i + 1}: '{chunks[i]}'")
    print(f"Chunk {i + 2}: '{chunks[i + 1]}'")
    print()

Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'
Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'
Chunk 4: 'is sentence five. This is sentence six.'



### What I noticed

This experiment shows me that the separator and chunk size both affect where the chunks end.

Even a small change in these settings can change the final chunks.

That is why chunking is not something I want to choose randomly in a RAG system.


## 3. Token-Based Splitting

So far I have been working with character-based chunk sizes.

Now I’m trying TokenTextSplitter.

Here the chunk size is based on tokens instead of characters.

This is useful to understand because language models work with tokens, so token-based limits can sometimes be more directly related to model limits.


In [27]:
# Method 3: Token-based splitting
print("\n3️⃣ TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=50,  # Size in tokens (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


3️⃣ TOKEN TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


## Comparing the Splitting Methods

Now I want to compare the three methods instead of looking at them separately.

I’m mainly looking at how each method decides the chunk boundaries and when each one may be useful.

I’m not trying to say that one method will always work best.

The right choice will depend on the type of data and the RAG system I’m building.


In [28]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")
print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  Use when: Working with token-limited models


In [29]:
print("TEXT SPLITTING METHODS COMPARISON")

print("\nCharacterTextSplitter:")
print("  - Simple and predictable")
print("  - Useful when the text has clear separators")
print("  - Can break text at less useful places")

print("\nRecursiveCharacterTextSplitter:")
print("  - Tries different separators")
print("  - Can keep the text structure better")
print("  - Good general-purpose option")

print("\nTokenTextSplitter:")
print("  - Works with token-based chunk sizes")
print("  - Useful when token limits matter")
print("  - Helps connect chunk size with model limits")

TEXT SPLITTING METHODS COMPARISON

CharacterTextSplitter:
  - Simple and predictable
  - Useful when the text has clear separators
  - Can break text at less useful places

RecursiveCharacterTextSplitter:
  - Tries different separators
  - Can keep the text structure better
  - Good general-purpose option

TokenTextSplitter:
  - Works with token-based chunk sizes
  - Useful when token limits matter
  - Helps connect chunk size with model limits


## My Takeaway

From these experiments, I can see that chunking is more than just setting a chunk size.

The separator, overlap, chunk size, and splitting method can all change the final result.

I also understand now why I need to look at the actual chunks instead of only checking how many chunks were created.

For the next part of my RAG work, I want to test these ideas with different types of documents and later compare them using embeddings and retrieval quality.
